# Bird classification project: EDA

This project's objective is to create a bird species classification model using the NABirds dataset. This is a collection of about 48000 photographs of at least 550 North American bird species found on deeplake. This dataset has images of various birds with labels for the bird categories and boxes to localize the bird in each image. 

This notebook's aim is to explore the dataset and propose image processing techniques to ease and optimize the classification process. 

Notebook structure: 
- Imports and setup
- Image corruption verification
- Duplicate image examination
- Image channel examination
- Image size examination
- Image class distribution
- Image visualization with bounding box
- Image quality analysis (mean class tracking) 
    - Brightness and contrast
    - Blur and sharpness (variance of the Laplacian) 
    - Color distribution
    - Background complexity with edge tensity 
    - Texture (entropy and local standard deviation) 
    - Correlations between metrics
- Inter-Class Similarity
- Image Transformations 
- EDA conclusions

# Imports and setup

In [ ]:
import random
from collections import Counter
import deeplake
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from collections import defaultdict
import phik
from matplotlib.ticker import PercentFormatter
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import umap.umap_ as umap
from python_files.eda import *

Setting random seed for reproducibility: 

In [ ]:
SEED = 42
rng = np.random.default_rng(SEED)
random.seed(SEED)

Loading the training and validation data: 

In [ ]:
ds_train = deeplake.load("hub://activeloop/nabirds-dataset-train", read_only=True)
ds_val = deeplake.load("hub://activeloop/nabirds-dataset-val", read_only=True)

Building a class label index: 

In [ ]:
LABEL_INDEX_PATH = Path("label_index_train.npz")
if LABEL_INDEX_PATH.exists():
    label_index = load_label_index(LABEL_INDEX_PATH)
else:
    label_index = build_label_index(ds_train)
    save_label_index(label_index, LABEL_INDEX_PATH)
print(f"Cached label index entries: {len(label_index)} classes")

In total, there are 555 classes. Now, creating and id_to_name mapping for the 555 classes: 

In [ ]:
class_names = ds_train.labels.info.get("class_names", None)
actual_labels_used = set(label_index.keys())

if class_names is not None:
    id_to_name = {
        label_id: class_names[label_id]
        for label_id in actual_labels_used
        if label_id < len(class_names)
    }
else:
    id_to_name = {label_id: str(label_id) for label_id in actual_labels_used}

print(f"Created id_to_name mapping with {len(id_to_name)} species")

# Image corruption verification

It is essential to check for any corrupted files before proceeding. The entire dataset is examined here, to have a trustworthy dataset before modeling. Simultaneously, any images that are all black, all white, or that are nearly uniform (variance less than one) are identified: 

In [ ]:
train_quality = check_image_quality(ds_train, name="train")
val_quality = check_image_quality(ds_val, name="validation")

print("\n" + "=" * 50)
print("QUALITY CHECK SUMMARY")
print("=" * 50)
print(
    f"Train:      {n_problem_images(train_quality)} problematic images / {train_quality['total']} checked"
)
print(
    f"Validation: {n_problem_images(val_quality)} problematic images / {val_quality['total']} checked"
)

No corrupt images or low variance images are seen in this dataset. 

# Duplicate images

Identifying exact duplicate images is also important to ensure that the model will be as accurate as possible. This is done by computing perceptual hashes to detect visually similar images (with the same pHash), and then md5 hashes on any visually similar images to see if they are exact duplicates. This ensures that the training data is not accidentally inflated with repeated images. 

In [ ]:
phash_groups_train = compute_phash_groups(ds_train)
phash_groups_val = compute_phash_groups(ds_val)

print(f"Found {len(phash_groups_train)} pHash duplicate groups in training data")
print(f"Found {len(phash_groups_val)} pHash duplicate groups in validation data")

print(
    f"Total images in these groups (train): {sum(len(v) for v in phash_groups_train.values())}"
)
print(
    f"Total images in these groups (val): {sum(len(v) for v in phash_groups_val.values())}"
)

In the training data, 72 duplicate pHashes have been found, and 99 duplicate pHash groups have been found in the validation data. Are these images exact duplicates? 

In [ ]:
exact_pairs_train, near_groups_train = refine_with_md5(ds_train, phash_groups_train)

print(f"Exact duplicate pairs (pixel-identical): {len(exact_pairs_train)}")
print(f"Near-duplicate groups (same pHash, different MD5): {len(near_groups_train)}")
print(f"Total near-duplicate images: {sum(len(v) for v in near_groups_train.values())}")

exact_pairs_val, near_groups_val = refine_with_md5(ds_val, phash_groups_val)
print(f"Exact duplicate pairs (pixel-identical): {len(exact_pairs_val)}")
print(f"Near-duplicate groups (same pHash, different MD5): {len(near_groups_val)}")
print(f"Total near-duplicate images: {sum(len(v) for v in near_groups_val.values())}")

## Duplicate image visualization

None of the similar images have exactly the same md5 hash. Looking at 10 near duplicate groups in the training dataset with the same label: 

In [ ]:
show_duplicate_groups(
    ds_train,
    near_groups_train,
    group_type="consistent",
    title_prefix="Train",
    id_to_name=id_to_name,
    n_groups=5,
)

Now, looking at validation groups with the same pHash: 

In [ ]:
show_duplicate_groups(
    ds_val,
    near_groups_val,
    group_type="consistent",
    title_prefix="Validation",
    id_to_name=id_to_name,
    n_groups=5,
)

From visual inspection, images with duplicate pHashes and the same label are essentially the same image, and will be dropped. First, examining training images with the same pHash and different labels: 

In [ ]:
show_duplicate_groups(
    ds_train,
    near_groups_train,
    group_type="inconsistent",
    title_prefix="Train",
    id_to_name=id_to_name,
    n_groups=10,
)

Finally, examining images from the validation set with different labels and the same pHash: 

In [ ]:
show_duplicate_groups(
    ds_val,
    near_groups_val,
    group_type="inconsistent",
    title_prefix="Val",
    id_to_name=id_to_name,
    n_groups=5,
)

All the training images with the same pHash value but with different labels are clearly different birds. Two of the same pHash values with different labels in the validation set show the same bird, which is indicative of a labelling error. 

## Dropping exact duplicate images: 

Now, all duplicates with the same label and same pHash are dropped. To do this, groups with the same pHash and label are identified, as are the indices to drop: 

In [ ]:
train_to_drop = get_same_label_duplicates_to_drop(ds_train, near_groups_train)
val_to_drop = get_same_label_duplicates_to_drop(ds_val, near_groups_val)

print(f"Training:   dropping {len(train_to_drop)} duplicate images")
print(f"Validation: dropping {len(val_to_drop)} duplicate images")

Now, a clean label index is created and saved, which excludes the dropped duplicates: 

In [ ]:
label_index_clean = {}
for label_id, indices in label_index.items():
    clean = np.array(
        [idx for idx in indices if idx not in train_to_drop], dtype=np.int64
    )
    if len(clean) > 0:
        label_index_clean[label_id] = clean

original_train = sum(len(v) for v in label_index.values())
cleaned_train = sum(len(v) for v in label_index_clean.values())
print(
    f"Training: {original_train:,} → {cleaned_train:,} (removed {original_train - cleaned_train})"
)

label_index_val = defaultdict(list)
for i, sample in enumerate(ds_val):
    label = int(sample["labels"].numpy().flat[0])
    label_index_val[label].append(i)
label_index_val = {k: np.array(v, dtype=np.int64) for k, v in label_index_val.items()}

label_index_val_clean = {}
for label_id, indices in label_index_val.items():
    clean = np.array([idx for idx in indices if idx not in val_to_drop], dtype=np.int64)
    if len(clean) > 0:
        label_index_val_clean[label_id] = clean

original_val = sum(len(v) for v in label_index_val.values())
cleaned_val = sum(len(v) for v in label_index_val_clean.values())
print(
    f"Validation: {original_val:,} → {cleaned_val:,} (removed {original_val - cleaned_val})"
)

save_label_index(label_index_clean, Path("label_index_train_clean.npz"))
save_label_index(label_index_val_clean, Path("label_index_val_clean.npz"))
print("\nSaved: label_index_train_clean.npz, label_index_val_clean.npz")

This filtered index will be used for the rest of the project to avoid duplicate images. 

# Image channels 

Image channels (grayscale/rbg) are next examined, as are average channel values and standard deviation: 

In [ ]:
sample_n = len(ds_train)
take = set(rng.choice(len(ds_train), size=sample_n, replace=False))

num_non_rgb = 0
means_c, stds_c = [], []

for i, sample in enumerate(ds_train):
    if i not in take:
        continue
    img = sample["images"].numpy()
    if img.ndim != 3 or img.shape[2] != 3:
        num_non_rgb += 1
        continue
    arr = img.astype(np.float32) / 255.0
    means_c.append(arr.mean(axis=(0, 1)))
    stds_c.append(arr.std(axis=(0, 1)))

print(f"Checked {len(take)}: non-RGB: {num_non_rgb}")
if means_c:
    mc = np.vstack(means_c)
    sc = np.vstack(stds_c)
    print("Channel means (R,G,B) mean:", mc.mean(0))
    print("Channel stds  (R,G,B) mean:", sc.mean(0))

All images in the sample have 3 channels (RGB). The green channel is the brightest channel, and the blue channel is the darkest with the highest amount of variation. The dataset appears to be well formed with normal color distribution and variation, and should be compatible with pretrained CNN backbones. 

Also checking a sample of the validation data: 

In [ ]:
sample_n = 1000
take = set(rng.choice(len(ds_val), size=sample_n, replace=False))

num_non_rgb = 0
means_c, stds_c = [], []

for i, sample in enumerate(ds_val):
    if i not in take:
        continue
    img = sample["images"].numpy()
    if img.ndim != 3 or img.shape[2] != 3:
        num_non_rgb += 1
        continue
    arr = img.astype(np.float32) / 255.0
    means_c.append(arr.mean(axis=(0, 1)))
    stds_c.append(arr.std(axis=(0, 1)))

print(f"Checked {len(take)}: non-RGB: {num_non_rgb}")
if means_c:
    mc = np.vstack(means_c)
    sc = np.vstack(stds_c)
    print("Channel means (R,G,B) mean:", mc.mean(0))
    print("Channel stds  (R,G,B) mean:", sc.mean(0))

The same observations are seen in the validation data.

# Image sizes

Before selecting an input image size and processing strategy it is essential to examine raw image dimensions. To begin, looking at the dataset summary: 

In [ ]:
for split, ds in [("train", ds_train), ("val", ds_val)]:
    print(f"\n== {split.upper()} ==")
    print(ds)
    print(ds.summary())
    print("tensors:", ds.tensors)

The training set has almost 24 000 images with heights ranging from 100-1024 pixels and widths ranging from 90-1024 pixels. The validation set has about 24 600 images with heights ranging from 98-1024 pixels and widths ranging from 117-1024 pixels. Resizing images will be necessary before modeling, as pytorch expects images that are the same size. In addition, fully connected CNN layers expect images of the same size. 

Images are of type uint8, which indicates that pixel values range from 0-255. Images are stored as jpeg files. Each image has a corresponding label of type integer and bouding box surrounding the bird, which could be useful for cropping purposes. 

Examining image size and aspect ratio in more detail:

In [ ]:
H_train, W_train, aspects_train, areas_train = analyze_dimensions(
    ds_train, sample_size=sample_n
)
H_val, W_val, aspects_val, areas_val = analyze_dimensions(ds_val, sample_size=sample_n)

In [ ]:
dim_df = pd.DataFrame(
    {
        "Value": np.concatenate([H_train, W_train, H_val, W_val]),
        "Dimension": ["Height"] * len(H_train)
        + ["Width"] * len(W_train)
        + ["Height"] * len(H_val)
        + ["Width"] * len(W_val),
        "Dataset": ["Train"] * len(H_train)
        + ["Train"] * len(W_train)
        + ["Val"] * len(H_val)
        + ["Val"] * len(W_val),
    }
)

plt.figure(figsize=(10, 5))
sns.boxplot(data=dim_df, y="Dimension", x="Value", hue="Dataset", orient="h")
plt.title("Image Dimension Distribution: Train vs Validation")
plt.xlabel("Pixels")
plt.legend(title="Dataset")
plt.show()

Generally, images are wider than they are high. Most images have width between 800-1000 pixels, with height around 600-800 pixels. Examining image aspect ratio and area: 

In [ ]:
colors = plt.cm.tab10.colors
TRAIN_COLOR = colors[0]
VAL_COLOR = colors[1]

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

# --- Aspect Ratio Distribution ---
axes[0, 0].hist(
    aspects_train, bins=50, alpha=0.5, density=True, label="Train", color=TRAIN_COLOR
)
axes[0, 0].hist(
    aspects_val, bins=50, alpha=0.5, density=True, label="Val", color=VAL_COLOR
)
axes[0, 0].axvline(
    np.median(aspects_train),
    color=TRAIN_COLOR,
    linestyle=":",
    label=f"Train median: {np.median(aspects_train):.2f}",
)
axes[0, 0].axvline(
    np.median(aspects_val),
    color=VAL_COLOR,
    linestyle=":",
    label=f"Val median: {np.median(aspects_val):.2f}",
)
axes[0, 0].set_title("Aspect Ratio Distribution")
axes[0, 0].set_xlabel("Width / Height")
axes[0, 0].set_ylabel("Density")
axes[0, 0].legend(fontsize=8)

# --- Width vs Height Scatter ---
axes[1, 0].scatter(W_train, H_train, alpha=0.5, s=10, label="Train", color=TRAIN_COLOR)
axes[1, 0].scatter(W_val, H_val, alpha=0.5, s=10, label="Val", color=VAL_COLOR)
axes[1, 0].plot(
    [0, max(max(W_train), max(W_val))],
    [0, max(max(W_train), max(W_val))],
    "k--",
    label="Square",
)
axes[1, 0].set_title("Width vs Height")
axes[1, 0].set_xlabel("Width (px)")
axes[1, 0].set_ylabel("Height (px)")
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(True, alpha=0.5)

train_pct = get_aspect_percent(aspects_train)
val_pct = get_aspect_percent(aspects_val)

x = np.arange(len(train_pct))
width = 0.35
axes[0, 1].bar(
    x - width / 2, train_pct.values, width, label="Train", alpha=0.8, color=TRAIN_COLOR
)
axes[0, 1].bar(
    x + width / 2, val_pct.values, width, label="Val", alpha=0.8, color=VAL_COLOR
)
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(train_pct.index, rotation=45)
axes[0, 1].set_title("Aspect Ratio Categories")
axes[0, 1].set_ylabel("Percentage (%)")
axes[0, 1].set_ylim(0, 100)
axes[0, 1].legend(fontsize=8)

# --- Image Area Distribution ---
axes[1, 1].hist(
    areas_train / 1e6,
    bins=50,
    alpha=0.5,
    density=True,
    label="Train",
    color=TRAIN_COLOR,
)
axes[1, 1].hist(
    areas_val / 1e6, bins=50, alpha=0.5, density=True, label="Val", color=VAL_COLOR
)
axes[1, 1].axvline(
    np.median(areas_train) / 1e6,
    color=TRAIN_COLOR,
    linestyle=":",
    label=f"Train median: {np.median(areas_train)/1e6:.2f} MP",
)
axes[1, 1].axvline(
    np.median(areas_val) / 1e6,
    color=VAL_COLOR,
    linestyle=":",
    label=f"Val median: {np.median(areas_val)/1e6:.2f} MP",
)
axes[1, 1].set_title("Image Area Distribution")
axes[1, 1].set_xlabel("Area (megapixels)")
axes[1, 1].set_ylabel("Density")
axes[1, 1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(
    f"\nTrain: median aspect={np.median(aspects_train):.2f}, median area={np.median(areas_train)/1e6:.2f} MP"
)
print(
    f"Val:   median aspect={np.median(aspects_val):.2f}, median area={np.median(areas_val)/1e6:.2f} MP"
)

The aspect ratio distribution and categories plots show that around 80% of images in the sample are wider than they are tall (landscape orientation), though around 10% of the sample images are in portrait mode, and a smaller proportion of images are square or very wide. The width vs height plot also shows that very few images are square, and that most points fall below the diagonal, which indicates that they are wider than they are tall. The most common width seems to be around 1000 pixels. A fairly wide range of image sizes are shown, which means that images sizes should be standardized before feeding them into a neural network. Resizing images to a square input will likely introduce some distortion for most of the images, and so bounding box cropping, center cropping, padding or rectangular resizing could be examined. 

The image area distribution plot shows the total pixel area distribution (height*width) for each image, and shows the image resolution size. Most images have an area of 0.7 megapixels, though the range is quite wide (from 0.2-1 megapixels). The variety in image sizes and resolutions show that images were collected from a range of sources and devices. In general, no appreciable difference is seen in terms of image sizes between training and validation data. 

# Class counts and distribution

This section analyzes class distribution, which can influence model bias, training stability, and generalization. 
First, computing class counts for both training and validation sets:

In [ ]:
class_names = ds_train.labels.info.get("class_names", None)
id_to_name = (
    {i: class_names[i] for i in range(len(class_names))} if class_names else None
)

train_counts = counts_from_index(label_index)
val_counts = Counter(int(sample["labels"].numpy()[0]) for sample in ds_val)

print(f"Train classes: {len(train_counts)}, images: {sum(train_counts.values())}")
print(f"Val classes:   {len(val_counts)}, images: {sum(val_counts.values())}")

Both the training and validation datasets have 555 classes. Are any classes in only the training set or only the validation set? 

In [ ]:
train_only = set(train_counts) - set(val_counts)
val_only = set(val_counts) - set(train_counts)
both = set(train_counts) & set(val_counts)
print(f"Classes in both: {len(both)}")
print(f"Classes only in train: {len(train_only)}")
print(f"Classes only in val:   {len(val_only)}")

The training and validation datasets share the same classes. Class imbalance is examined next, as the distribution of examples per class directly impacts model learning, evaluation fairness, and choice of metric. To quantify class distribution, the Gini coefficient which ranges from 0 (completely balanced) to 1 (very imbalanced) and measures the overall shape of the class distribution is calculated. In addition, the imbalance ratio (largest class size/smallest class size) and the coverage by the top 10% of classes are measured. 

In [ ]:
df = pd.DataFrame(
    {"class_id": list(train_counts.keys()), "train_count": list(train_counts.values())}
).set_index("class_id")

df["val_count"] = df.index.map(val_counts).fillna(0).astype(int)
df["total_count"] = df["train_count"] + df["val_count"]
df["train_pct"] = df["train_count"] / df["train_count"].sum()
df["total_pct"] = df["total_count"] / df["total_count"].sum()
df = df.sort_values("train_count", ascending=False)
if id_to_name:
    df.insert(0, "class_name", df.index.map(id_to_name))

imbalance_ratio = df["train_count"].max() / df["train_count"].min()
gini_train = gini(df["train_count"])
top10pct_classes = max(1, int(0.1 * len(df)))
coverage_top10pct = (
    df["train_count"].head(top10pct_classes).sum() / df["train_count"].sum()
)

print(f"Imbalance ratio (max/min): {imbalance_ratio:.1f}:1")
print(f"Gini (train counts): {gini_train:.3f}")
print(f"Top 10% classes cover {coverage_top10pct*100:.1f}% of train images")
print("Top 10 classes:")
display(df.head(10))
print("Bottom 10 classes:")
display(df.tail(10))

Overall, this dataset does not exhibit extreme imbalance. Quite a few classes have 60 examples, which is the maximum. Some variation is shown between the species with the largest number of examples and the rarest species (white winged dark-eyed Junco), as the imbalance ratio is 15:1. The relatively low Gini coefficient (0.192) and low class coverage (only 13.8% of training images are in the top 10% of classes) indicates that the classes are relatively balanced. It is also important to note that one class in the training set (Dark-eyed Junco) has only 4 classes, and is therefore at a disadvantage for few shot modeling when the training set will be adjusted to having 5 images per class. 

The class distribution is now visualized with a histogram and a scatter plot of train vs validation classes: 

In [ ]:
train_counts_arr = df["train_count"]
val_counts_arr = df["val_count"]
colors = plt.cm.tab10.colors
TRAIN_COLOR, VAL_COLOR = colors[0], colors[1]

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

axes[0].hist(
    train_counts_arr,
    bins=40,
    alpha=0.5,
    label=f"Train (n={len(train_counts_arr)})",
    color=TRAIN_COLOR,
    edgecolor="black",
)
axes[0].hist(
    val_counts_arr,
    bins=40,
    alpha=0.5,
    label=f"Val (n={len(val_counts_arr)})",
    color=VAL_COLOR,
    edgecolor="black",
)
axes[0].axvline(
    np.median(train_counts_arr),
    color=TRAIN_COLOR,
    linestyle="--",
    lw=2,
    label=f"Train median: {np.median(train_counts_arr):.0f}",
)
axes[0].axvline(
    np.median(val_counts_arr),
    color=VAL_COLOR,
    linestyle="--",
    lw=2,
    label=f"Val median: {np.median(val_counts_arr):.0f}",
)
axes[0].set_title("Distribution of Images per Class")
axes[0].set_xlabel("Number of Images per Class")
axes[0].set_ylabel("Number of Classes")
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

common_ids = list(set(train_counts.keys()) & set(val_counts.keys()))
train_common = [train_counts.get(c, 0) for c in common_ids]
val_common = [val_counts.get(c, 0) for c in common_ids]

axes[1].scatter(train_common, val_common, alpha=0.5, s=20, color="steelblue")
max_val = max(max(train_common), max(val_common)) * 1.1
axes[1].plot([0, max_val], [0, max_val], "r--", label="Equal distribution")
axes[1].set_title("Train vs Validation Counts per Class")
axes[1].set_xlabel("Training Samples")
axes[1].set_ylabel("Validation Samples")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)


plt.tight_layout()
plt.show()

These two plots show that the training and validation class distributions are similar, as the histogram shows similar number of classes for the training and validation data, and the scatterplot shows that most classes are grouped towards the center diagonal line. Some outliers in the train vs validation counts plot are present (where many training samples are present but not many validation samples are present, and vice versa). In general, the distribution shows a long left tail, which indicates that few classes have few samples (less than 20), and most of the classes have between 30-60 images. Most of the classes (around 140, or a quarter of all classes) have 60 images per class. 

# Image visualization

Images in the dataset can now be visually examined, with bounding boxes surrounding the birds. First, rare training classes are observed with bounding boxes that are part of the dataset:

## Rare classes

In [ ]:
rare_ids = df.tail(5).index.tolist()
show_samples_by_class(
    ds_train,
    rare_ids,
    "Rare classes (train)",
    label_index=label_index_clean,
    per_class=3,
    show_boxes=True,
    id_to_name=id_to_name,
)

Next, rare validation classes are shown: 

In [ ]:
rare_ids = df.sort_values("val_count", ascending=False).tail(4).index.tolist()
show_samples_by_class(
    ds_val,
    rare_ids,
    "Rare classes (val)",
    label_index=label_index_val_clean,
    per_class=3,
    show_boxes=True,
    id_to_name=id_to_name,
)

## Common classes

Next, common training classes are shown: 

In [ ]:
common_ids = df.sort_values("train_count", ascending=False).index.tolist()[:5]
show_samples_by_class(
    ds_train,
    common_ids,
    "Common classes (train)",
    label_index=label_index_clean,
    per_class=3,
    show_boxes=True,
    id_to_name=id_to_name,
)

Common validation classes are visualized: 

In [ ]:
common_ids = df.sort_values("val_count", ascending=False).index.tolist()[:4]
show_samples_by_class(
    ds_val,
    common_ids,
    "Common classes (val)",
    label_index=label_index_val_clean,
    per_class=3,
    show_boxes=True,
    id_to_name=id_to_name,
)

## Median classes

5 classes around the median are also shown to get a representative sample, starting with training data: 

In [ ]:
median_idx = len(df) // 2
mid_freq_ids = df.sort_values("train_count").index.tolist()[
    median_idx - 2 : median_idx + 3
]
show_samples_by_class(
    ds_train,
    mid_freq_ids,
    "Mid-frequency classes (train)",
    label_index=label_index_clean,
    per_class=3,
    show_boxes=True,
    id_to_name=id_to_name,
)

5 classes around the median are also visualized for validation data: 

In [ ]:
mid_freq_ids_val = df.sort_values("val_count").index.tolist()[
    median_idx - 2 : median_idx + 3
]
show_samples_by_class(
    ds_val,
    mid_freq_ids_val,
    "Mid-frequency classes (val)",
    label_index=label_index_val_clean,
    per_class=3,
    show_boxes=True,
    id_to_name=id_to_name,
)

## Overall image observations: 

- No notable differences are seen between common and rare images, or between the training and validation sets in terms of quality, bird pose, or image orientation. 
- Quite a bit of lighting variation is seen in these images, with some being quite bright (for example, images taken during a sunny day with a sky or water background), and some being darker (taken in a forest with more shadows). Dominant colors in this dataset include shades of green, gray, brown, and blue. 
- Image background complexity has a lot of variation, from a simple uniform background (sky or water) to complex backgrounds with branches, trees, foliage, or plants. 
- Image resolution seems to also have some variation, where some images are very clear and show many details, and some show less fine details. Some focus variation is seen, where few birds are blurry, especially those with water backgrounds. 
- A few images also have writing (likely of the photographer) on them. 
- Images are not all the same size, though most are in landscape mode (as seen earlier), and no huge differences in size are seen. 
- Images from the same species are clearly recognized as similar birds. Some images from different classes have similar bird shapes or colors. For example, small birds such as the Boreal chickadee and the House Sparrow or ducks such as Barrow’s Goldeneye and Harlequin Duck could be easily confused. 


Image orientation and pose variation observations: 
- In general, a profile of a bird is shown (side view). Quite a lot of variation is shown with bird orientation, where some images show birds with a front or back/angled view. Birds are seen flying, swimming, diving, or standing. Occasionally, occlusions such as rocks or branches are blocking part of the bird. 
- Birds are often centered in the image, with varied size and scale. Sometimes, only part of the bird is seen (head or body). 
- The provided bounding boxes properly surround all birds examined. 


Implications for modeling: 
- Raw images present lighting variation, so varying the color jitter, and applying normalization could help a model to learn the importance of shape and texture. 
- Images also have some blur and variations in resolution, which indicates that a model must be robust to image details and quality. Including some image sharpening, smoothing, or random Gaussian Blur could help with model generalization. 
- Geometric augmentations might also be helpful, as birds are seen from many different orientations and poses. Using techniques such as image flips, rotations, cropping could improve model generalization to birds of the same species. Moreover, using RandomResizedCrop could be helpful, as birds in this dataset appear at various different scales.  
- Some birds blend in with backgrounds, and backgrounds are quite variable. Therefore, using the provided bounding boxes to crop the image could be helpful, as this could also eliminate some of the complex background effects. 
- Some species (owls, ducks, small birds) look visually similar, which means that a model which can capture fine details will be essential, such as ResNet50. Center cropping using the bounding box could also be helpful here, as this will  focus the image on a bird’s details. 



## Bounding box coverage analysis

To further inform a potential image cropping strategy, the bounding box area can be compared to the full image area to see how much of the image the bird occupies. For example, if the bird occupies less than 30% of the total image area, image cropping is likely a good strategy. However, if over 60% of the image is occupied by the bird; cropping is likely unnecessary and resizing the image can be used. 

In [ ]:
n_samples = 1000
coverages_train = bbox_coverage_simple(ds_train, n_samples=n_samples, seed=SEED)
coverages_val = bbox_coverage_simple(ds_val, n_samples=n_samples, seed=SEED)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
bp = ax.boxplot(
    [coverages_train * 100, coverages_val * 100],
    tick_labels=["Train", "Validation"],
    patch_artist=True,
    widths=0.5,
)

colors = plt.cm.tab10.colors
bp["boxes"][0].set_facecolor(colors[0])
bp["boxes"][1].set_facecolor(colors[1])

med_train = np.median(coverages_train) * 100
med_val = np.median(coverages_val) * 100
n_small_train = (coverages_train < 0.30).sum() / len(coverages_train) * 100
n_small_val = (coverages_val < 0.30).sum() / len(coverages_val) * 100

summary = (
    f"Train: median={med_train:.1f}%, small={n_small_train:.1f}%\n"
    f"Val:   median={med_val:.1f}%, small={n_small_val:.1f}%"
)
ax.text(
    0.98,
    0.02,
    summary,
    transform=ax.transAxes,
    fontsize=10,
    va="bottom",
    ha="right",
    bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
    family="monospace",
)
for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(1)

ax.set_ylabel("Bounding Box Coverage (%)")
ax.set_title("Bird Size in Frame: Train vs Validation")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

This plot shows that over half of both training and validation images sampled have birds that take up less than 30% of the image's area. Therefore, bounding box cropping will likely be used as a preprocessing technique. Most bounding boxes take up between 20-40% of the entire image.

# Image Quality Analysis

This section analyzes image characteristics across the dataset to understand:
- **Brightness & Contrast** - Lighting conditions and range
- **Blur/Sharpness** - Image focus quality (variance of the Laplacian)
- **Color Distribution** - RGB channel statistics for color bias detection
- **Background Complexity** - Edge density as a proxy for cluttered backgrounds
- **Texture Features** - Entropy and local standard deviation for texture variation

These metrics help identify:
- Classes with systematically poor image quality
- Need for preprocessing (histogram equalization, sharpening, etc.)
- Potential data augmentation strategies

To begin, metrics are computed for the training and validation data, selecting 15 samples from each class: 

In [ ]:
train_metrics_df = compute_image_metrics_raw(
    ds_train, label_index_clean, sample_per_class=15, seed=SEED
)
train_metrics_df["split"] = "train"

In [ ]:
val_metrics_df = compute_image_metrics_raw(
    ds_val, label_index_val_clean, sample_per_class=15, seed=SEED
)
val_metrics_df["split"] = "val"

Then, the training and validation dataframes are combined to have all sample information in one place. Any samples with |z scores| over 2.5 are included as outliers to keep track of extreme samples for each metric in each class: 

In [ ]:
all_metrics_df = pd.concat([train_metrics_df, val_metrics_df], ignore_index=True)

print(
    f"Total samples: {len(all_metrics_df)} "
    f"(Train: {len(train_metrics_df)}, Val: {len(val_metrics_df)})"
)
print(
    f"Classes: Train={train_metrics_df['class_id'].nunique()}, "
    f"Val={val_metrics_df['class_id'].nunique()}"
)

metric_cols = [
    "brightness",
    "contrast",
    "blur",
    "r_mean",
    "g_mean",
    "b_mean",
    "edge_density",
    "entropy",
    "highfreq_std",
]

all_metrics_df = add_zscores_by_split(all_metrics_df, metric_cols, z_thresh=2.5)
all_metrics_df.head()

Slightly more validation samples are present than training samples, as the training set had some classes with less than 15 samples. All classes have been represented in the image metric calculations. Each image metric can now be examined one by one: 


## Brightness & Contrast Analysis

**Brightness** (mean grayscale 0-1): Measures overall image luminance. 
- Low values indicate dark/underexposed images
- High values indicate bright/overexposed images

Examining brightness distribution: 

In [ ]:
plot_metric_box_and_hist_seaborn(
    all_metrics_df,
    metric="brightness",
    train_color=TRAIN_COLOR,
    val_color=VAL_COLOR,
    metric_label="Brightness (0–1)",
    bins=30,
    thresholds=None,
)

The brightness distribution appears to be normally shaped, with similar range between the train and validation data. Some outlier images are present with low brightness (<0.2) and high brightness (>0.8). Examining examples of dark, light, and medium brightness images: 

In [ ]:
show_metric_examples_low_mid_high(
    all_metrics_df,
    metric="brightness",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    k=3,
    title="BRIGHTNESS: 3 Darkest, 3 Typical, 3 Brightest",
    metric_label="Brightness",
)

The very light images appear a little blurry, though birds are still clear. All the brightest birds are seen flying in an almost white sky, and all the darkest birds are images taken at night. In the very dark images, bird shape and features can get lost in the background (for example, with the Burrowing Owl). Briefly examining the classes with highest and lowest average brightness: 

In [ ]:
brightness_split_summary = plot_top_bottom_classes_by_split(
    all_metrics_df,
    metric="brightness",
    id_to_name=id_to_name,
    n_classes=3,
    metric_label="Brightness (0–1)",
)

show_class_examples_for_split(
    df=all_metrics_df,
    metric="brightness",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    class_ids_train_bottom=brightness_split_summary["train_bottom_ids"],
    class_ids_train_top=brightness_split_summary["train_top_ids"],
    class_ids_val_bottom=brightness_split_summary["val_bottom_ids"],
    class_ids_val_top=brightness_split_summary["val_top_ids"],
    add_spacing=True,
    title="Brightness: Example Images by Class (Train vs Val)",
)

Average darkest and brightest classes are not the same across the training and validation data, which indicates that images in classes have variability in their brightness levels. Two owl species are among the darkest classes, and two swift and kite species are among the brightest classes. Again, most images from brighter classes show birds flying. 

### Contrast analysis: 

**Contrast** (standard deviation of grayscale 0-1): Measures dynamic range.
- Low values indicate flat, low-contrast images
- High values indicate high dynamic range
- Affects feature discriminability

Examining image contrast: 

In [ ]:
plot_metric_box_and_hist_seaborn(
    all_metrics_df,
    metric="contrast",
    train_color=TRAIN_COLOR,
    val_color=VAL_COLOR,
    metric_label="Contrast (0–1)",
    bins=30,
    thresholds=None,
)

Image contrast also has an almosst normally shaped distribution, with a slight right tail. The training and validation distributions also look very similar. Examining some low, high, and typical contrast images: 

In [ ]:
show_metric_examples_low_mid_high(
    all_metrics_df,
    metric="contrast",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    k=3,
    title="CONTRAST: 3 Lowest, 3 Typical, 3 Highest",
    metric_label="Contrast",
)

From this visualization, low contrast images are generally showing birds flying in a uniform background. Medium contrast images have some darker and high contrast images are generally showing birds in trees during the day, where the tree is dark and the background is quite light. 

In [ ]:
contrast_split_summary = plot_top_bottom_classes_by_split(
    all_metrics_df,
    metric="contrast",
    id_to_name=id_to_name,
    n_classes=3,
    metric_label="Contrast (0–1)",
)

show_class_examples_for_split(
    df=all_metrics_df,
    metric="contrast",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    class_ids_train_bottom=contrast_split_summary["train_bottom_ids"],
    class_ids_train_top=contrast_split_summary["train_top_ids"],
    class_ids_val_bottom=contrast_split_summary["val_bottom_ids"],
    class_ids_val_top=contrast_split_summary["val_top_ids"],
    add_spacing=True,
    title="Contrast: Example Images by Class (Train vs Val)",
)

Swift species are consistently among those with the lowest contrast in both the training and validation data, and are all seen flying in this sample. 2 woodpecker and owl species are among those with higher contrast, and are all shown with backgrounds containing trees. 

Examining the relationship between brightness and contrast: 

In [ ]:
plot_brightness_vs_contrast(
    all_metrics_df, train_color=TRAIN_COLOR, val_color=VAL_COLOR
)

No clear relationship is seen between brightness and contrast. Most images in the sample have average brightness between 0.3-0.7 and contrast between 0.06-0.27. 

## Variance of the Laplacian (blur/shaprness score)

This is a proxy of image sharpness by measuring edges (high variance). 
- **Low values**: blurry, out-of-focus images, and reduces features that could differentiate between classes
- **High values**: sharp, well-focused images

This metric helps identify:
- Classes with consistently blurry images (could need sharpening augmentation),
- Motion blur issues (birds in flight), and
- Issues with focus. 

Examining the distribution: 

In [ ]:
plot_metric_box_and_hist_seaborn(
    all_metrics_df,
    metric="blur",
    train_color=TRAIN_COLOR,
    val_color=VAL_COLOR,
    metric_label="Variance of Laplacian",
    bins=40,
    thresholds=None,
)

This distribution is highly skewed to the right, with few images having very high sharpness. Both the training and validation data have long right tails, though the validation data has more outliers with high Laplacian variance. Examining some sharp, typical, and blurry images: 

In [ ]:
show_metric_examples_low_mid_high(
    all_metrics_df,
    metric="blur",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    k=3,
    title="BLUR: 3 Lowest (sharp), 3 Typical, 3 Highest (blurry)",
    metric_label="Blur",
)

Blurriest images are birds seen in flight, as capturing a fast moving subject sometimes results in a blurry image. Typical images show a in-focus bird with a clear or blurry background. The sharpest images show all details of a bird with its background. Examining the sharpest and blurriest classes:

In [ ]:
blur_res = plot_top_bottom_classes_by_split(
    all_metrics_df,
    metric="blur",
    id_to_name=id_to_name,
    n_classes=3,
    metric_label="Blur (variance of Laplacian)",
)
show_class_examples_for_split(
    df=all_metrics_df,
    metric="blur",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    class_ids_train_bottom=blur_res["train_bottom_ids"],
    class_ids_train_top=blur_res["train_top_ids"],
    class_ids_val_bottom=blur_res["val_bottom_ids"],
    class_ids_val_top=blur_res["val_top_ids"],
    add_spacing=True,
    title="BLUR: Example Images by Class (Train vs Val)",
)

Similarly to contrast, swift species also are the blurriest images, likely due to movement in flight and lack of background. No consistent classes are among the sharpest, as this could be influenced by factors such as background, camera quality, and focus. 

## Color Distribution Analysis

Analyzing per-channel (R, G, B) mean to detect:
- If the training and validation data have similar color balance, 
- If any color bias exists (images that are predominantly one color), 
- How color relates to backgrounds. 

Bird species often have different colors or distinct colored features, and so color is a key discriminative feature for this dataset. Examining the mean color distributions: 

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
train_mask = all_metrics_df["split"] == "train"
val_mask = all_metrics_df["split"] == "val"

train_r = all_metrics_df.loc[train_mask, "r_mean"]
train_g = all_metrics_df.loc[train_mask, "g_mean"]
train_b = all_metrics_df.loc[train_mask, "b_mean"]
val_r = all_metrics_df.loc[val_mask, "r_mean"]
val_g = all_metrics_df.loc[val_mask, "g_mean"]
val_b = all_metrics_df.loc[val_mask, "b_mean"]

n_train = train_mask.sum()
n_val = val_mask.sum()

# --- Boxplots ---
bp = axes[0, 0].boxplot(
    [train_r, val_r, train_g, val_g, train_b, val_b],
    tick_labels=["R(Train)", "R(Val)", "G(Train)", "G(Val)", "B(Train)", "B(Val)"],
    patch_artist=True,
    widths=0.6,
    medianprops=dict(color="black", linewidth=1.5),
)
colors_box = [TRAIN_COLOR, VAL_COLOR] * 3
for patch, color in zip(bp["boxes"], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[0, 0].set_title("RGB Channel Means: Train vs Val", fontsize=12, fontweight="bold")
axes[0, 0].set_ylabel("Channel Mean (0–1)")
axes[0, 0].grid(True, axis="y", alpha=0.3)
axes[0, 0].axhline(0.5, color="gray", linestyle="--", alpha=0.3)

# --- Histogram (Train) ---
axes[0, 1].hist(
    train_r,
    bins=30,
    alpha=0.6,
    color="red",
    label="Red",
    edgecolor="black",
    linewidth=0.3,
    weights=np.ones(len(train_r)) / len(train_r) * 100,
)
axes[0, 1].hist(
    train_g,
    bins=30,
    alpha=0.6,
    color="green",
    label="Green",
    edgecolor="black",
    linewidth=0.3,
    weights=np.ones(len(train_g)) / len(train_g) * 100,
)
axes[0, 1].hist(
    train_b,
    bins=30,
    alpha=0.6,
    color="blue",
    label="Blue",
    edgecolor="black",
    linewidth=0.3,
    weights=np.ones(len(train_b)) / len(train_b) * 100,
)
axes[0, 1].set_title(
    "TRAIN: RGB Channel Mean Distributions", fontsize=12, fontweight="bold"
)
axes[0, 1].set_xlabel("Channel Mean (0–1)")
axes[0, 1].set_ylabel("Relative Frequency (%)")
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# --- Histogram (Val) ---
axes[1, 1].hist(
    val_r,
    bins=30,
    alpha=0.6,
    color="red",
    label="Red",
    edgecolor="black",
    linewidth=0.3,
    weights=np.ones(len(val_r)) / len(val_r) * 100,
)
axes[1, 1].hist(
    val_g,
    bins=30,
    alpha=0.6,
    color="green",
    label="Green",
    edgecolor="black",
    linewidth=0.3,
    weights=np.ones(len(val_g)) / len(val_g) * 100,
)
axes[1, 1].hist(
    val_b,
    bins=30,
    alpha=0.6,
    color="blue",
    label="Blue",
    edgecolor="black",
    linewidth=0.3,
    weights=np.ones(len(val_b)) / len(val_b) * 100,
)
axes[1, 1].set_title(
    "VAL: RGB Channel Mean Distributions", fontsize=12, fontweight="bold"
)
axes[1, 1].set_xlabel("Channel Mean (0–1)")
axes[1, 1].set_ylabel("Relative Frequency (%)")
axes[1, 1].legend(fontsize=9)
axes[1, 1].grid(True, alpha=0.3)

# --- Dominant Channel per Image ---
tmp = all_metrics_df.copy()
tmp["dominant"] = tmp[["r_mean", "g_mean", "b_mean"]].idxmax(axis=1)
tmp["dominant"] = tmp["dominant"].map(
    {"r_mean": "Red", "g_mean": "Green", "b_mean": "Blue"}
)

dom_counts = tmp.groupby(["split", "dominant"]).size().unstack(fill_value=0)
dom_counts = dom_counts.reindex(columns=["Red", "Green", "Blue"])
dom_pct = dom_counts.div(dom_counts.sum(axis=1), axis=0) * 100

x = np.arange(2)
width = 0.25
axes[1, 0].bar(
    x - width,
    dom_pct.loc[["train", "val"], "Red"],
    width,
    color="red",
    alpha=0.7,
    label="Red",
)
axes[1, 0].bar(
    x,
    dom_pct.loc[["train", "val"], "Green"],
    width,
    color="green",
    alpha=0.7,
    label="Green",
)
axes[1, 0].bar(
    x + width,
    dom_pct.loc[["train", "val"], "Blue"],
    width,
    color="blue",
    alpha=0.7,
    label="Blue",
)
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(["Train", "Val"])
axes[1, 0].set_title("Dominant Channel per Image", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("Relative Frequency (%)")
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, axis="y", alpha=0.3)

plt.suptitle("COLOR DISTRIBUTION OVERVIEW", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

rgb_std = all_metrics_df[["r_mean", "g_mean", "b_mean"]].std(axis=1)
n_grayscale = (rgb_std < 0.02).sum()
print(
    f"\nNear-grayscale images (R ≈ G ≈ B): {n_grayscale} ({100*n_grayscale/len(all_metrics_df):.1f}%)"
)

In general, the blue channel has a larger distribution than the red or green channels. No difference is seen between the training and validation distributions, and a higher proportion of images have red channel dominance. In addition, about 22% of images in the sample have similar R/G/B channel values. Examining typical images with high red, green, blue channels, or near grayscale images: 

In [ ]:
fig, axes = plt.subplots(4, 6, figsize=(15, 10))

df_color = all_metrics_df.copy()
df_color["r_bias"] = df_color["r_mean"] - (df_color["g_mean"] + df_color["b_mean"]) / 2
df_color["g_bias"] = df_color["g_mean"] - (df_color["r_mean"] + df_color["b_mean"]) / 2
df_color["b_bias"] = df_color["b_mean"] - (df_color["r_mean"] + df_color["g_mean"]) / 2
df_color["rgb_std"] = df_color[["r_mean", "g_mean", "b_mean"]].std(axis=1)

df_color = df_color[(df_color["brightness"] > 0.15) & (df_color["brightness"] < 0.85)]

np.random.seed(SEED)

for i, (split, ds) in enumerate([("train", ds_train), ("val", ds_val)]):
    df_split = df_color[df_color["split"] == split].nlargest(10, "r_bias")
    samples = df_split.sample(min(3, len(df_split)))
    for j, (_, row) in enumerate(samples.iterrows()):
        show_img(
            axes[0, i * 3 + j],
            ds,
            int(row["idx"]),
            f"{split.title()}: R-bias={row['r_bias']:.2f}",
        )

for i, (split, ds) in enumerate([("train", ds_train), ("val", ds_val)]):
    df_split = df_color[df_color["split"] == split].nlargest(10, "g_bias")
    samples = df_split.sample(min(3, len(df_split)))
    for j, (_, row) in enumerate(samples.iterrows()):
        show_img(
            axes[1, i * 3 + j],
            ds,
            int(row["idx"]),
            f"{split.title()}: G-bias={row['g_bias']:.2f}",
        )

for i, (split, ds) in enumerate([("train", ds_train), ("val", ds_val)]):
    df_split = df_color[df_color["split"] == split].nlargest(10, "b_bias")
    samples = df_split.sample(min(3, len(df_split)))
    for j, (_, row) in enumerate(samples.iterrows()):
        show_img(
            axes[2, i * 3 + j],
            ds,
            int(row["idx"]),
            f"{split.title()}: B-bias={row['b_bias']:.2f}",
        )

for i, (split, ds) in enumerate([("train", ds_train), ("val", ds_val)]):
    df_split = df_color[df_color["split"] == split].nsmallest(10, "rgb_std")
    samples = df_split.sample(min(3, len(df_split)))
    for j, (_, row) in enumerate(samples.iterrows()):
        show_img(
            axes[3, i * 3 + j],
            ds,
            int(row["idx"]),
            f"{split.title()}: std={row['rgb_std']:.3f}",
        )

for i, label in enumerate(
    ["Red-Biased", "Green-Biased", "Blue-Biased", "Near-Grayscale"]
):
    axes[i, 0].annotate(
        label,
        xy=(-0.3, 0.5),
        xycoords="axes fraction",
        fontsize=11,
        fontweight="bold",
        rotation=90,
        va="center",
    )

plt.suptitle(
    "Color Bias & Grayscale Examples (Train | Val)", fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

Images with higher red channel values indeed have red, orange or yellow backgrounds. Images with higher green channel mean values are taken around foliage, and images with higher blue channel mean values are mostly taken with the sky as a background. Grayscale images are generally taken with water as a background, from this relatively small sample. Examining dominant channels averaged per class gives: 

In [ ]:
if "r_bias" not in all_metrics_df.columns:
    all_metrics_df["r_bias"] = (
        all_metrics_df["r_mean"]
        - (all_metrics_df["g_mean"] + all_metrics_df["b_mean"]) / 2
    )
    all_metrics_df["g_bias"] = (
        all_metrics_df["g_mean"]
        - (all_metrics_df["r_mean"] + all_metrics_df["b_mean"]) / 2
    )
    all_metrics_df["b_bias"] = (
        all_metrics_df["b_mean"]
        - (all_metrics_df["r_mean"] + all_metrics_df["g_mean"]) / 2
    )


color_bias_res = plot_top_classes_by_color_bias(all_metrics_df, id_to_name, n_classes=3)
show_color_bias_class_examples(
    all_metrics_df, ds_train, ds_val, id_to_name, color_bias_res
)

In general, the top 3 red species contain the burrowing owl and two sparrow species. The top 3 green biased classes contain warblers in the validation data, but a variation of classes for the training data. Larger birds such as hawks, falcons, or eagles are seen in the top 3 blue classes. This could be an indication that the habitat or surroundings of an image is indicative or correlated with the species (ex: possibly large birds such as Hawks are more frequently seen flying with a blue sky background, or warblers are generally found in trees surrounded by foliage). 

## Background Complexity (Edge Density)

This metric measures the amount of edges present in an image. To do this, the image is converted to grayscale and Sobel gradients are applied. The magnitude of these gradients are calculated, and the 75th percentile is used as a threshold. Edge density is the fraction of pixels with strong edges (Sobel filter > threshold).

- **High edge density**: images with lots of edges, and indicate a complex background (ex: branches, leaves, forest)
- **Low edge density**: images with few edges, and indicate a simple background, such as sky, water or blur. 


Implications for classification:
- High background complexity may confuse the model, and bounding box cropping may be necessary. Background blurring augmentations could also be used as preprocessing techniques. 

In [ ]:
plot_metric_box_and_hist_seaborn(
    all_metrics_df,
    metric="edge_density",
    train_color=TRAIN_COLOR,
    val_color=VAL_COLOR,
    metric_label="Edge Density (background complexity)",
    bins=30,
    thresholds=None,
)

This distribution is highly skewed to the left, as most images have an edge density around 0.25, and less than 25 of the sampled images have edge density less than this. Some images have an edge density of 0, which indicates that they do not have significant edges (at least 75% of the image does not have edges). This could be the case where an image has a solid background with a small bird. Examining some of the images with 0, median, and high edge density: 

In [ ]:
show_metric_examples_low_mid_high(
    all_metrics_df,
    metric="edge_density",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    k=3,
    title="EDGE DENSITY: 3 Lowest, 3 Typical, 3 Highest",
    metric_label="Edge Density",
)

Indeed, these images with 0 edge density generally all have solid backgrounds, and seem to have high brightness. The rest of the sample images all have edge density of 0.25, and show more complex backgrounds. Examining edge density by species:

In [ ]:
edge_res = plot_top_bottom_classes_by_split(
    all_metrics_df,
    metric="edge_density",
    id_to_name=id_to_name,
    n_classes=3,
    metric_label="Edge Density",
)

show_class_examples_for_split(
    df=all_metrics_df,
    metric="edge_density",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    class_ids_train_bottom=edge_res["train_bottom_ids"],
    class_ids_train_top=edge_res["train_top_ids"],
    class_ids_val_bottom=edge_res["val_bottom_ids"],
    class_ids_val_top=edge_res["val_top_ids"],
    add_spacing=True,
    title="EDGE DENSITY: Example Images by Class (Train vs Val)",
)

The training and validation classes for highest and lowest edge density do not agree with each other, as most classes have average texture complexity (0.25). No large variation is seen between the average highest/lowest edge densities either, which indicates that most images have average texture complexity.

## Texture Analysis (Entropy & High Frequency Standard Deviation)

Entropy measures the amount of randomness or diversity of intensity values in an image, and ranges from 0-8. 
- **High entropy** indicates that an image has diverse pixel values, complex textures (feathers, patterns, trees or branches), where
- **Low entropy** indicate an image with uniform regions and simple textures. 

If entropy is consistent across the dataset, this indicates that images have a similar quality. Moreover, texture analysis and histogram diversity is important for this dataset, as birds of different species often have different texture patterns and colors in terms of plumage. Examining entropy's distribution across the sample: 

In [ ]:
plot_metric_box_and_hist_seaborn(
    all_metrics_df,
    metric="entropy",
    train_color=TRAIN_COLOR,
    val_color=VAL_COLOR,
    metric_label="Entropy (bits)",
    bins=30,
    thresholds=None,
)

Similarly to edge density, this distribution is quite skewed to the left, with most images being fairly complex (entropy between 6-8), and a few images having low complexity. Both the training and validation distributions are very similar. Examining some images with high, medium, and low entropy: 

In [ ]:
show_metric_examples_low_mid_high(
    all_metrics_df,
    metric="entropy",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    k=3,
    title="ENTROPY: 3 Lowest, 3 Typical, 3 Highest",
    metric_label="Entropy",
)

As expected, low entropy images show a uniform background with the bird as the only feature, and do not have a lot of color variation. Medium entropy images (about 7) show birds with a more complex background (on water, in a tree), while birds with entropy values close to 8 show a lot of detail in the bird and its background, with color and brightness variations. Examining entropy by classes: 

In [ ]:
entropy_res = plot_top_bottom_classes_by_split(
    all_metrics_df,
    metric="entropy",
    id_to_name=id_to_name,
    n_classes=3,
    metric_label="Entropy (bits)",
)
show_class_examples_for_split(
    df=all_metrics_df,
    metric="entropy",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    class_ids_train_bottom=entropy_res["train_bottom_ids"],
    class_ids_train_top=entropy_res["train_top_ids"],
    class_ids_val_bottom=entropy_res["val_bottom_ids"],
    class_ids_val_top=entropy_res["val_top_ids"],
    add_spacing=True,
    title="ENTROPY: Example Images by Class (Train vs Val)",
)

Similarly to edge density, classes with overall low entropy include swifts or the swallow tailed kite, with images of birds flying against a plain sky. Examining image local standard deviation: 

### High frequency standard deviation

This metric indicates the variation from the Gaussian smoothed local mean. A high high frequency standard deviation indicates that an image has detailed patterns and more local edges, while a low value indicates smoother image texture overall. Examining this distribution: 

In [ ]:
plot_metric_box_and_hist_seaborn(
    all_metrics_df,
    metric="highfreq_std",
    train_color=TRAIN_COLOR,
    val_color=VAL_COLOR,
    metric_label="High-frequency Std Dev",
    bins=30,
    thresholds=None,
)

This distribution is skewed to the right, where most images have high frequency standard deviation between 5-10, and some images have higher values (above 15). No significant differences are seen between the training and validation distributions. Examining some images with low, median, and high high frequency standard deviations: 

In [ ]:
show_metric_examples_low_mid_high(
    all_metrics_df,
    metric="highfreq_std",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    k=3,
    title="High-frequency Std Dev: 3 Lowest, 3 Typical, 3 Highest",
    metric_label="High-frequency Std Dev",
)

Similarly to entropy, images with low values are blurry, with a uniform background. Images with high frequency standard deviation around the median show more texture, expeically with the image's background (grass, waves, branches). Images with higher values are quite high resolution with detailed backgrounds, where individual blades of grass, tree bark, or needles are evident. Examining this by class: 

In [ ]:
hfstd_res = plot_top_bottom_classes_by_split(
    all_metrics_df,
    metric="highfreq_std",
    id_to_name=id_to_name,
    n_classes=3,
    metric_label="Local Std (high-frequency)",
)
show_class_examples_for_split(
    df=all_metrics_df,
    metric="highfreq_std",
    ds_train=ds_train,
    ds_val=ds_val,
    id_to_name=id_to_name,
    class_ids_train_bottom=hfstd_res["train_bottom_ids"],
    class_ids_train_top=hfstd_res["train_top_ids"],
    class_ids_val_bottom=hfstd_res["val_bottom_ids"],
    class_ids_val_top=hfstd_res["val_top_ids"],
    add_spacing=True,
    title="LOCAL STD: Example Images by Class (Train vs Val)",
)

Swifts are often seen flying, and are once again categorized as a species with low texture, as the background is generally uniform. The great horned owl is typically photographed in trees, which contributes to higher high frequency standard deviation due to the complexity of a forest background. This, as well as the entropy observations suggest that image texture metrics may reflect on a specie's habitat and typical photography conditions, which could confuse a classifier. Examining the relationship between entropy and high frequency standard deviation: 

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

for split, color, label in [
    ("train", TRAIN_COLOR, "Train"),
    ("val", VAL_COLOR, "Val"),
]:
    df_split = all_metrics_df[all_metrics_df["split"] == split]
    ax.scatter(
        df_split["entropy"],
        df_split["highfreq_std"],
        alpha=0.5,
        s=25,
        color=color,
        label=label,
    )

ax.set_title("Entropy vs High Frequency Std Dev (Train vs Val)")
ax.set_xlabel("Entropy (histogram diversity)")
ax.set_ylabel("High-frequency Std Dev (texture variation)")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()

These two variables appear to have a positive correlation. Images with higher entropy have more chances of also having higher high frequency standard deviation, indicating that some images with high tonal variation also have more variation in texture.

## Correlations: 

Finally, a phi-k correlation between the variables extracted from images (mean brightness, standard deviation of brightness, variance of the laplacian, local standard deviation, and entropy) will be examined: 

In [ ]:
num_cols = [
    "brightness",
    "contrast",
    "blur",
    "edge_density",
    "entropy",
    "highfreq_std",
    "r_mean",
    "g_mean",
    "b_mean",
]

corr_df = all_metrics_df.copy()
corr_df["label"] = corr_df["class_id"].astype("category")
phik_input = corr_df[num_cols].replace([np.inf, -np.inf], np.nan).dropna()
phik_overview = phik_input.phik_matrix(interval_cols=num_cols, njobs=1)

plt.figure(figsize=(16, 10))
mask = np.triu(np.ones_like(phik_overview, dtype=bool))
ax = sns.heatmap(
    phik_overview,
    annot=True,
    mask=mask,
    cmap="Blues",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": r"$\Phi_k$ correlation"},
)
ax.set_title(r"$\Phi_k$ Correlation Heatmap", fontsize=18, pad=14)
plt.tight_layout()
plt.show()

Key correlations observed:
- Brightness and RGB channels (0.87-0.99): Brightness is derived from RGB channels, so these features are strongly correlated. In addition, all channels are strongly correlated with other (0.77-0.95). 
- Blur and high frequency standard deviation: these metrics both measure image sharpness, and are redundant. 
- Entropy is moderately correlated with all metrics except for the Laplacian variance, as it is a good general score for image complexity. The independance from the Laplacian variance indicates that image blur/sharpness and complexity are independent. 

This image quality analysis examined brightness, contrast, the Laplacian variance, color distribution, and texture features from around 15 randomly sampled images per class for 550 classes. Key findings include: 
- Brightness showed a relatively normal distribution with most brightness values between 0.4-0.6. The darkest images were taken at night, and brightest images showed birds flying agains a uniform sky. 
- Contrast was slightly skewed to the right, with most images having values between 0.12-0.20. Low contrast images showed birds (swifts) flying in a uniform sky background, and high contrast images showed woodpecker or owl species in trees during the day. 
- The Laplacian variance had a right skewed distribution. The blurriest images showed swifts in flight, and the sharpest images showed more detailed backgrounds (trees, grass, water). 
- Color distribution analysis showed the red channel was slightly more dominant than the other channels, and the blue channel had a slightly wider distribution than the other channels. Images with green biased channels generally showed birds flying or birds in the water, and images with green biased channels showed birds around foliage. 
- The edge density distribution was highly left skewed, with some images having values of 0 (birds flying against uniform sky background). Images with high edge density generally had more complex backgrounds (trees/shrubs). 
- Entropy also had a left skewed distribution with low entropy images showing uniform backgrounds with Swifts or Kites, and high entropy images had more complex backgrounds with birds in trees or foliage. 
- The high frequency standard deviation distribution was skewed to the right, and showed highly detailed images of birds (such as owls) in complex habitats (grass, detailed tree bark) for high values. 

In general, no difference was found between the training and validation images or distributions. Certain species are photographed in specific environments. For example, species such as Swifts or Kites are often shown flying against a uniform sky (low contrast, low entropy, low edge density, high brightness). Owls tend to be photographed at night or during the day in the forest, where images have lower brightness, high contrast and high edge density. Other species are likely to be identified by their surroundings as well. Bounding box cropping is a technique that can reduce this background influence on species. Techniques such as CLAHE (Contrast Limited Adaptive Histogram Equalization) could be used to enhance low contrast images (ex: birds in flight), and colour augmentation could be used to mitigate species related colour biases. As there could be correlations with bird species and image complexity, augmentations such as random cropping or random erasing could be useful. These augmentations would have the objective to ensure that the model learns the bird morphology rather than species backgrounds. 

# Inter-Class Similarity

Now, a pretrained ResNet-50 model can be used to identify classes that will likely be difficult to classify during few shot learning. To do this, each image's feature vector is computed. Then, each class is summarized using the class's mean feature vector. The cosine similarity between classes indicates classes that are similar in the feature space. These similar classes are identified, and UMAP is used to visualize the class space. 

Some initial variables are defined, insuring that the GPU will be used, and initializing the feature extractor: 

In [ ]:
SAMPLES_PER_CLASS = 10
MAX_UMAP_POINTS = 8000
TOP_SIMILAR_PAIRS = 20

device = get_device()
print("Using device:", device)
extractor = FeatureExtractor(device)
print("Feature extractor ready (embedding dim:", extractor.embedding_dim, ")")

Now for each species:
-  A 2048 dimensional embedding are obtained for a sample after it has passed through the ResNet-50 feature extractor. 
- The embeddings are then averaged per class to obtain a class prototype, which summarizes general features for the species (colors, textures, and shapes). 

In [ ]:
embeddings, labels, prototypes, proto_class_ids = extract_class_embeddings(
    ds_train,
    label_index,
    extractor,
    samples_per_class=SAMPLES_PER_CLASS,
    seed=SEED,
)

Next, a class similarity matrix is computed using cosine similarity on the class prototypes, and the top 20 species (which correspond to the species that are closest in the ResNet-50 feature space) are extracted: 

In [ ]:
similarity_df = compute_class_similarity(prototypes, proto_class_ids)
print("Similarity matrix shape:", similarity_df.shape)

similar_pairs = get_most_similar_pairs(similarity_df, n_pairs=TOP_SIMILAR_PAIRS)

The distribution of these similarity values is examined now, by plotting a histogram of the similarity scores of all unique class pairs: 

In [ ]:
sim_vals = similarity_df.values[np.triu_indices_from(similarity_df.values, k=1)]

fig, ax = plt.subplots(figsize=(8, 5))
n, bins, patches = ax.hist(sim_vals, bins=40, edgecolor="black", alpha=0.7)

ax.yaxis.set_major_formatter(PercentFormatter(xmax=len(sim_vals)))
ax.set_ylabel("Percent of class pairs")

ax.axvline(
    x=np.mean(sim_vals),
    color="red",
    linestyle="--",
    label=f"Mean: {np.mean(sim_vals):.3f}",
)
ax.set_xlabel("Cosine similarity between class prototypes")
ax.set_title("Distribution of inter-class similarities (ResNet50 prototypes)")
ax.legend()
plt.show()

print(f"Min similarity:   {sim_vals.min():.4f}")
print(f"Max similarity:   {sim_vals.max():.4f}")

This graph shows the spread of cosine similarities for all the unique species pairs, where values close to 0 indicate visually distinct classes, and values clost to 1 indicate highly similar classes. This distribution is unimodal with a right tail. Most classes have cosine similarity values between about 0.1-0.4, which indicates that most classes are visually separable (not very similar). The amount of class pairs decreases slowly between cosine similarity values of about 0.5, and very few class pairs have cosine similarity over 0.8. These classes with high cosine similarity are likely hard to distinguish from one another as they likely have similar species shapes, colors, poses, or image backgrounds. 

Visualizing the top 5 most similar classes: 

In [ ]:
for _, row in similar_pairs.head(5).iterrows():
    show_class_pair_examples(
        ds_train,
        label_index,
        cls_a=row["class_1"],
        cls_b=row["class_2"],
        id_to_name=id_to_name,
        similarity=row["similarity"],
        n_per_class=3,
        seed=SEED,
    )

All of the top 5 most similar species are subspecies of the same species class (ex: Carolina Chickadee and Black-capped Chickadee are both Chickadees). Upon visual inspection, these classes with the highest cosine similarity scores are also very visually similar. Birds in these classes have similar poses, shapes, are generally centered, and have similar colours. Looking at a list of the top 20 species: 

In [ ]:
print(f"\nTOP {TOP_SIMILAR_PAIRS} MOST SIMILAR CLASS PAIRS:")
for _, row in similar_pairs.iterrows():
    c1 = row["class_1"]
    c2 = row["class_2"]
    name1 = id_to_name.get(c1, f"Class {c1}")
    name2 = id_to_name.get(c2, f"Class {c2}")
    print(f"  {c1} ({name1}) ↔ {c2} ({name2}): similarity = {row['similarity']:.4f}")

From this, it is possible to observe that out of all bird species in this dataset, it is expected to be most challenging to correctly classify chickadee species, hummingbird species, the California and Gambel's Quail, the Greater and Lesser Yellowlegs, and Sandpipers and Sanderlings. 

UMAP (Uniform Manifold Approximation and Projection) with a cosine distance metric is also used to visualize the 2048-dimensional ResNet-50 embeddings in 2 dimensions: 

In [ ]:
if len(embeddings) > MAX_UMAP_POINTS:
    idx_umap = rng.choice(len(embeddings), size=MAX_UMAP_POINTS, replace=False)
    emb_for_umap = embeddings[idx_umap]
    labels_for_umap = labels[idx_umap]
else:
    emb_for_umap = embeddings
    labels_for_umap = labels

print(f"\nFitting UMAP on {len(emb_for_umap)} image embeddings...")

reducer = umap.UMAP(
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=SEED,
)

umap_coords = reducer.fit_transform(emb_for_umap)
print("UMAP coords shape:", umap_coords.shape)

unique_classes = sorted(np.unique(labels_for_umap))
n_classes = len(unique_classes)
colors = plt.cm.nipy_spectral(np.linspace(0, 1, n_classes))
class_to_color = {cid: colors[i] for i, cid in enumerate(unique_classes)}
pt_colors = [class_to_color[cid] for cid in labels_for_umap]

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(umap_coords[:, 0], umap_coords[:, 1], c=pt_colors, s=5, alpha=0.6)
ax.set_title(f"UMAP of ResNet50 embeddings ({n_classes} classes)")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.set_xticks([])
ax.set_yticks([])
plt.show()

Each point in this graph corresponds to a single image, with species label as the color. Points appearing close together in this graph are visually similar images, and well separated clusters show groups that are easier to 
separate from the others. This plot shows a large cluster with mostly red points, but also containing green, blue, and yellow points. This suggests that these speices are harder to classify. Some parts of the plot are more elongated, which indicates more variation in the species. 

Further examining UMAP on average class embeddings (class prototypes), with labels for some more distinct, typical, and generic classes: 

In [ ]:
if "class_distinctiveness" not in globals():
    mean_sim_to_others = []
    for cid in proto_class_ids:
        row = similarity_df.loc[cid].drop(cid)
        mean_sim_to_others.append(row.mean())

    class_distinctiveness = (
        pd.DataFrame(
            {
                "class_id": proto_class_ids,
                "mean_similarity_to_others": mean_sim_to_others,
            }
        )
        .sort_values("mean_similarity_to_others")
        .reset_index(drop=True)
    )

num_each = 3
most_distinct = class_distinctiveness.head(num_each)["class_id"].tolist()
most_generic = class_distinctiveness.tail(num_each)["class_id"].tolist()
global_mean_sim = class_distinctiveness["mean_similarity_to_others"].mean()
class_distinctiveness["abs_diff_from_global_mean"] = (
    class_distinctiveness["mean_similarity_to_others"] - global_mean_sim
).abs()

typical_candidates = class_distinctiveness.sort_values("abs_diff_from_global_mean")

already_picked = set(most_distinct + most_generic)
typical_normal = [
    cid for cid in typical_candidates["class_id"].tolist() if cid not in already_picked
][:num_each]

print("Most DISTINCT:", most_distinct)
print("Most GENERIC:", most_generic)
print("Typical:", typical_normal)

highlight_ids = most_distinct + most_generic + typical_normal

print("\nFitting UMAP on class prototypes...")

reducer_proto = umap.UMAP(
    n_neighbors=min(15, len(prototypes) - 1),
    min_dist=0.1,
    metric="cosine",
    random_state=SEED,
)

umap_prototypes = reducer_proto.fit_transform(prototypes)
print("UMAP prototypes shape:", umap_prototypes.shape)

proto_colors = [class_to_color.get(cid, (0, 0, 0, 1.0)) for cid in proto_class_ids]
class_id_to_proto_idx = {cid: i for i, cid in enumerate(proto_class_ids)}

fig, ax = plt.subplots(figsize=(12, 10))
proto_colors = [class_to_color.get(cid, (0, 0, 0, 1.0)) for cid in proto_class_ids]
ax.scatter(
    umap_prototypes[:, 0], umap_prototypes[:, 1], c=proto_colors, s=45, alpha=0.85
)
for _, row in similar_pairs.head(10).iterrows():
    c1, c2 = row["class_1"], row["class_2"]
    if c1 in class_id_to_proto_idx and c2 in class_id_to_proto_idx:
        i1 = class_id_to_proto_idx[c1]
        i2 = class_id_to_proto_idx[c2]
        x1, y1 = umap_prototypes[i1]
        x2, y2 = umap_prototypes[i2]
        ax.plot(
            [x1, x2], [y1, y2], "-", color="red", alpha=0.5, linewidth=1.5, zorder=0
        )


for cid in highlight_ids:
    if cid in class_id_to_proto_idx:
        i = class_id_to_proto_idx[cid]
        x, y = umap_prototypes[i]

        species_name = id_to_name.get(cid, f"Class {cid}")
        short_name = species_name.split(",")[0]

        if cid in most_distinct:
            prefix = "Distinct"
        elif cid in most_generic:
            prefix = "Generic"
        else:
            prefix = "Typical"

        if cid == 499:
            dx, dy = 0.02, -0.04
        else:
            dx, dy = 0.02, 0.02

        ax.text(
            x + dx,
            y + dy,
            f"{prefix} #{cid}: {short_name}",
            fontsize=9,
            fontweight="bold",
            ha="left",
            va="bottom",
            bbox=dict(facecolor="white", alpha=0.7, edgecolor="none", pad=1),
        )


ax.set_title(
    "UMAP of Class Prototypes\n"
    "(Most distinctive / most generic / typical similarity classes labeled)",
    fontsize=14,
)
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

This UMAP plot shows the relationship between bird species, and each point represent one species. The clusters of points indicate classes that are likely difficult to classify (visually similar), where independent cluster show classes that are more visually distinct, and easier for classification tasks. The darker blobs in this image (green/blue/black) correspond to groups of species that are separated from the main red/orange/yellow blob on the upper right. These species, such as the Snowy Egret, Brown Pelican or Roseate Spoonbill are more visually distinct than the other species, while the red/yellow/light green species such as the Hermit Thrush, Rusty Blackbird, or Barn Swallow are more closely related visually, and likely more challenging to properly classify. 

Based on the clusters of high similarity pairs, it is likely a good idea to avoid strong color augmentations, as this could make the already similar species even harder to classify. In addition, these highly similar species will likely need to be classified via model fine tuning so that the model can learn small image differences. 


# Image Transformations

Now that the dataset has been explored, transformations and data augmentation strategies can be designed to address issues found in previous sections. Some augmentations could help the model to learn bird morphology instead of image background. 

EDA highlighted several challenges with this dataset:

| Challenge              | Example species     | Proposed  preprocessing / augmentation                   |
|------------------------|------------------------------------------|------------------------------------------------------------------------|
| Background dependency  | Swifts, kites (uniform sky)              | Bounding-box cropping, background-robust augmentation                 |
| Dark images            | Owls (night photography)                 | CLAHE, gamma correction                                               |
| Low contrast           | Birds in flight                          | CLAHE, contrast adjustment                                            |
| Blurry images          | Fast-moving birds                        | Sharpening, light deblurring                                         |
| Inter-class similarity | Highly similar pairs (e.g., hummingbirds, chickadees) | Geometric-heavy, color-conservative augmentation            |
| Low bird coverage      | Small birds in large backgrounds         | Bounding-box cropping with padding, aspect-preserving resizing        |

Now, the proposed preprocessing and autmentation techniques can be implemented, using Albumentations.




First, defining the final size and normalization variables for ImageNet: 

In [ ]:
IMG_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

Next, initializing preprocessing transformations for: 
- CLAHE (contrast limited adaptive histogram equalization): enhancing local contrast
- Gamma correction: brightness adjustments
- Sharpening: blur reduction
- Bounding box cropping: removes background according to the bounding box included in the dataset. 

In [ ]:
PREPROCESSING_TRANSFORMS = {
    "original": lambda img, ds, idx: img,
    "clahe": lambda img, ds, idx: apply_clahe(img),
    "gamma_brighten": lambda img, ds, idx: apply_gamma_correction(img, gamma=1.3),
    "sharpen": lambda img, ds, idx: apply_sharpening(img, amount=5, sigma=1.0),
    "bbox_crop": lambda img, ds, idx: apply_bbox_crop(img, ds, idx, padding_ratio=0.15),
}

print(f"Defined {len(PREPROCESSING_TRANSFORMS)} preprocessing transforms")

Before visualizing these transformations, some representative sample images are chosen from the training dataset: 
- Dark samples are in the lowest 5% in terms of brightness, 
- Low contrast samples are in the lowest 5% in terms of contrast,
- Blurry images are in the lowest 5% in terms of Laplacian variance, 
- Images with a complex background are in the highest 5% in terms of edge density, and 
- Normal images have brightness and contrast around the median. 

In [ ]:
test_cases = build_test_cases(all_metrics_df, id_to_name, n_samples=3)

print("Test cases by category:")
for category, samples in test_cases.items():
    print(f"  {category}: {len(samples)} images")
    for idx, cid, name in samples[:2]:
        print(f"    - idx={idx}, class={name[:30]}...")

Now that a sample has been selected, these preprocessing transformations can be visualized: 

In [ ]:
visualize_preprocessing(
    ds_train,
    test_cases,
    PREPROCESSING_TRANSFORMS,
    categories=["dark", "low_contrast", "blurry", "complex_background", "normal"],
)

Observations: 

For dark images: 
- CLAHE brightens shadowed regions, though having too high a value here could make background features stand out too much. 
- Gamma correction produces a smoother global image brightening
- Sharpening the image does not have a meaningful effect. 

For low contrast images: 
- These samples are taken of birds in the water, so CLAHE increases the effect of waves, as well as the bird’s outline and feathers. 
- Gamma brightening decreases the separation from the bird and the water, which could decrease model performance. This transformation is perhaps more useful for darker images only. 
- Image sharpening visually has no noticeable effect as compared to the original image. 

For blurry images: 
- CLAHE preprocessing increases the level of detail seen in the bird and the bird’s background. 
- Sharpening the blurry image has a minimal effect, showing perhaps slightly more detail. 

For images with complex backgrounds: 
- CLAHE further enhances a complex background, and would be best paired with cropping out the background so as not to add more complexity. 
- Brightening images with complex background reduces the effect of some dark shadows that can hide part of the bird. 

For normal images, 
- Simply resizing and normalization is sufficient, as these images don’t need extra enhancements. 

Overall, bounding box cropping zooms in on the bird and centres it, which is likely desirable so that the model can focus on the bird and not its background. Sharpening did not produce any dramatic visual effects. Brightening the image helped to add detail to darker images and areas with shadows, and CLAHE was useful for enhancing details in low contrast or blurry images. 

## Resize strategy comparison

As previously seen, images must be a uniform and square size before modeling. As the images in this dataset are different sizes with various aspect ratios, three strategies for image cropping/padding are explored to visually investigate which is best to preserve bird features and avoid image distortion: 
- Direct resizing (squish): distorts image
- LongestMaxSize with padding: preserves aspect ratio, and uses reflection padding 
- Bbox Crop with padding: crops to the provided bounding box and pads, also preserving aspect ratio. 

In [ ]:
rng = np.random.default_rng(SEED)
train_indices = all_metrics_df[all_metrics_df["split"] == "train"]["idx"].values
sample_indices = rng.choice(
    train_indices, size=min(5, len(train_indices)), replace=False
)
compare_resize_strategies(ds_train, sample_indices.tolist(), id_to_name=id_to_name)

Simple resizing (squish) induces image distortion, and will not be used. As expected, both other strategies keep the original aspect ratio, though the bird is sometimes reflected as part of the padding. Bounding box cropping nicely centres each bird so that the bird takes up more of the image, though it removes other image information (background). 

## Augmentation Pipelines using Albumentations

Now, various augmentation pipelines can be defined and visualized: 
- baseline: resize only
- standard: uses both geometric and color augmentations
- geometric_heavy: uses small color changes (some species are visually similar) and heavier geometric changes (rotation, shear, perspective)
- color_conservative: also uses small color changes with light geometric changes (only horizontal flips)
- background_robust: crops the image, includes cutout and noise to reduce model relying too heavily on the background
- quality_adaptive: targeted at low contrast or dark images, and uses CLAHE, modifies image brightness and contrast, and brightens/darkens the image. 

For each pipeline, a training version which normalizes the image, and a visualization version without normalization. 


In [ ]:
AUGMENTATION_PIPELINES = get_augmentation_pipelines(img_size=IMG_SIZE, for_viz=False)
AUGMENTATION_PIPELINES_VIZ = get_augmentation_pipelines(img_size=IMG_SIZE, for_viz=True)

BBOX_CROP_TRANSFORM = get_bbox_crop_transform(img_size=IMG_SIZE, for_viz=False)
BBOX_CROP_TRANSFORM_VIZ = get_bbox_crop_transform(img_size=IMG_SIZE, for_viz=True)

print("Training augmentation pipelines:")
for name, pipeline in AUGMENTATION_PIPELINES.items():
    print(f"  • {name}: {len(pipeline.transforms)} transforms")

print(
    f"\nBbox crop + aspect-preserving pipeline: {len(BBOX_CROP_TRANSFORM.transforms)} transforms"
)
print("\nVisualization augmentation pipelines:")
for name, pipeline in AUGMENTATION_PIPELINES_VIZ.items():
    print(f"  • {name}: {len(pipeline.transforms)} transforms")

Now, visualizing these pipelines: 

In [ ]:
viz_pipeline_fns = {}

for name, pipeline in AUGMENTATION_PIPELINES_VIZ.items():
    viz_pipeline_fns[name] = make_fn(pipeline)

viz_pipeline_fns["bbox_crop_pipeline"] = lambda img, ds, idx: BBOX_CROP_TRANSFORM_VIZ(
    image=apply_bbox_crop(img, ds, idx)
)["image"]

sample_indices_aug: list[int] = []
for cat in ["normal", "dark", "low_contrast", "blurry", "complex_background"]:
    if cat in test_cases:
        sample_indices_aug.extend([idx for idx, _, _ in test_cases[cat][:2]])

if len(sample_indices_aug) == 0:
    rng = np.random.default_rng(SEED)
    train_indices = all_metrics_df[all_metrics_df["split"] == "train"]["idx"].values
    sample_indices_aug = rng.choice(
        train_indices, size=min(4, len(train_indices)), replace=False
    ).tolist()

visualize_augmentations(
    ds_train,
    sample_indices_aug,
    viz_pipeline_fns,
    id_to_name=id_to_name,
    figsize_scale=2.5,
)

Based on these pipeline visualizations, the baseline pipeline is useful as a reference, as it simply resizes the images, preserving all colours and aspect ratio. The standard pipeline shows moderate colour and geometric variations in the image. The geometric heavy pipeline tends to crop off some of the birds, and will not be used in the modelling section. The colour conservative pipeline preserves image colours while including some light geometric augmentations. The background robust pipeline will not be used, as the random erasing tends to erase parts of the birds. It would be better to only erase areas outside of the bounding box. The quality adaptive pipeline increases contrast for low contrast images and increases image brightness for darker images. Geometry is similar to the original image. The bbox crop pipeline centers and magnifies the bird while preserving colours and geometry. 

While most of these pipelines will likely benefit some of the images in the dataset, the standard pipeline will be used during modelling as a starting point, as it is the pipeline most generalized to all images. The validation images will be resized and normalized only: 

In [ ]:
val_transform = A.Compose(
    [
        A.LongestMaxSize(max_size=IMG_SIZE),
        A.PadIfNeeded(
            min_height=IMG_SIZE,
            min_width=IMG_SIZE,
            border_mode=cv2.BORDER_REFLECT_101,
        ),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ]
)

train_transform = AUGMENTATION_PIPELINES["standard"]

print("Final transforms configured:")
print(
    f"  • Training:  {len(train_transform.transforms)} transforms (standard pipeline)"
)
print(f"  • Validation:{len(val_transform.transforms)} transforms (resize + normalize)")

# EDA conclusions

Overall, this EDA section has found that the NA birds dataset is comprised of 555 classes, split into training data (23928 samples) and validation data (24633 samples). 
This dataset does not have any corrupt images, and does have some duplicate images. Images have 3 channels (RGB) with normal colours and distributions. Images vary in size, with most being in landscape mode. This dataset is not highly imbalanced, nor is it completely balanced, as about 140 of the training classes have 60 samples each, and the sample number decreases for the rest of the classes. Bounding boxes and species labels are included in the dataset, which help to focus the image on the bird only. 

In general, images follow a normal brightness and contrast distribution. Some images (especially those of birds in flight) are fairly blurry, which could hamper model predictions in the following section. Some outlier images have an especially high Laplacian variance, and show very complex backgrounds with great detail. No colour bias is present, though some species could be distinguished by the colour of their background (ex: shrubs vs water/sky). Texture analysis showed most images to have complex features, as birds are often found in trees or shrubs. This section also showed that certain bird species could be identified by their backgrounds (ex: swifts tend to be seen flying, and owls tend to be seen in the forest). Moreover, species such as chickadee and hummingbird variations have high inter-class similarity and will likely be challenging to correctly classify. Some moderate geometric and color image transformations have been identified, and a standard method with light geometric and colour jitter will likely be used in the modelling section as a starting point, as this will not modify any of the images too much. 

In the next section, few shot modelling will be performed on this dataset using at most 5 images in each class. 